<a href="https://colab.research.google.com/github/abd500253-coder/Machine_Learning_Journey/blob/main/Power%20Transformer/Power_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Power Transformations to Improve Linear Models
In this notebook, we explore the use of `PowerTransformer` from `sklearn.preprocessing` to apply Box-Cox and Yeo-Johnson transformations on our dataset. Power transformations make data more Gaussian-like, which can stabilize variance and improve the performance of linear models like Linear Regression.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import warnings

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import PowerTransformer

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

## 1. Load and Explore the Data
We will use the Concrete Compressive Strength dataset. First, we load the data and inspect its shape, missing values, and statistical properties.

In [ ]:
# Load the dataset (update the path as needed)
df = pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/Data Sets/concrete_data.csv')

# Display first few rows
display(df.head())

# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Check dataset shape and info
print(f"\nDataset Shape: {df.shape}\n")
df.info()

# Summary statistics
display(df.describe())

## 2. Train-Test Split and Baseline Model
Before applying any transformations, let's separate our features (`X`) from our target (`y`), split the data, and train a baseline Linear Regression model to see our starting performance.

In [ ]:
# Separate features and target
X = df.drop(columns=['Strength'])
y = df['Strength']

# 80-20 Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline Linear Regression model
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

# Evaluate Baseline
print("Baseline R2 Score:", r2_score(y_test, y_pred))
print("Baseline Cross Validation Score:", np.mean(cross_val_score(lr, X, y, scoring='r2')))

## 3. Visualizing Original Distributions
Let's visualize the distribution and QQ plots of our features before any transformations are applied. A normal distribution aligns well with the red line on a QQ plot.

In [ ]:
# Plotting histograms and QQ plots
for col in X_train.columns:
    plt.figure(figsize=(16, 4))

    # Histogram
    plt.subplot(121)
    sns.histplot(X_train[col], kde=True)
    plt.title(f"{col} - Distribution")

    # QQ Plot
    plt.subplot(122)
    stats.probplot(X_train[col], dist="norm", plot=plt)
    plt.title(f"{col} - QQ Plot")

    plt.show()

## 4. Box-Cox Transformation
The Box-Cox transformation requires all data points to be strictly positive ($>0$). Because our dataset contains zero values (e.g., in Blast Furnace Slag or Fly Ash), we add a tiny constant ($0.000001$) to the features.

In [ ]:
# Initialize PowerTransformer for Box-Cox
pt_boxcox = PowerTransformer(method='box-cox')

# Apply transform (adding a small constant to handle 0 values)
X_train_boxcox = pt_boxcox.fit_transform(X_train + 0.000001)
X_test_boxcox = pt_boxcox.transform(X_test + 0.000001)
X_boxcox = pt_boxcox.fit_transform(X + 0.000001)

# Display calculated lambdas
print("Box-Cox Lambdas:")
display(pd.DataFrame({'cols': X_train.columns, 'box_cox_lambdas': pt_boxcox.lambdas_}))

# Train and evaluate on Box-Cox transformed data
lr.fit(X_train_boxcox, y_train)
y_pred_boxcox = lr.predict(X_test_boxcox)

print("Box-Cox R2 Score:", r2_score(y_test, y_pred_boxcox))
print("Box-Cox Cross Val Score:", np.mean(cross_val_score(lr, X_boxcox, y, scoring='r2')))

### Visualizing Box-Cox Results
Let's compare the feature distributions before and after the Box-Cox transformation.

In [ ]:
# Convert to DataFrame for plotting
X_train_boxcox_df = pd.DataFrame(X_train_boxcox, columns=X_train.columns)

for col in X_train_boxcox_df.columns:
    plt.figure(figsize=(14, 4))

    plt.subplot(121)
    sns.histplot(X_train[col], kde=True)
    plt.title(f"{col} - Before Box-Cox")

    plt.subplot(122)
    sns.histplot(X_train_boxcox_df[col], kde=True)
    plt.title(f"{col} - After Box-Cox")

    plt.show()

## 5. Yeo-Johnson Transformation
Unlike Box-Cox, the Yeo-Johnson transformation natively handles zero and negative values, so we do not need to add a constant. It is the default method in `sklearn`'s `PowerTransformer`.

In [ ]:
# Initialize PowerTransformer (defaults to Yeo-Johnson)
pt_yeo = PowerTransformer()

X_train_yeo = pt_yeo.fit_transform(X_train)
X_test_yeo = pt_yeo.transform(X_test)
X_yeo = pt_yeo.fit_transform(X)

# Train and evaluate on Yeo-Johnson transformed data
lr.fit(X_train_yeo, y_train)
y_pred_yeo = lr.predict(X_test_yeo)

print("Yeo-Johnson R2 Score:", r2_score(y_test, y_pred_yeo))
print("Yeo-Johnson Cross Val Score:", np.mean(cross_val_score(lr, X_yeo, y, scoring='r2')))

# Display calculated lambdas
print("\nYeo-Johnson Lambdas:")
display(pd.DataFrame({'cols': X_train.columns, 'Yeo_Johnson_lambdas': pt_yeo.lambdas_}))

### Visualizing Yeo-Johnson Results
Let's compare the feature distributions before and after the Yeo-Johnson transformation.

In [ ]:
# Convert to DataFrame for plotting
X_train_yeo_df = pd.DataFrame(X_train_yeo, columns=X_train.columns)

for col in X_train_yeo_df.columns:
    plt.figure(figsize=(14, 4))

    plt.subplot(121)
    sns.histplot(X_train[col], kde=True)
    plt.title(f"{col} - Before Yeo-Johnson")

    plt.subplot(122)
    sns.histplot(X_train_yeo_df[col], kde=True)
    plt.title(f"{col} - After Yeo-Johnson")

    plt.show()

## 6. Comparing Transform Lambdas
Finally, let's view the optimal $\lambda$ (lambda) values chosen by both the Box-Cox and Yeo-Johnson methods side-by-side to observe how they scaled each specific feature.

In [ ]:
# Side-by-side comparison of lambdas
lambda_comparison_df = pd.DataFrame({
    'Features': X_train.columns,
    'Box_Cox_Lambdas': pt_boxcox.lambdas_,
    'Yeo_Johnson_Lambdas': pt_yeo.lambdas_
})

display(lambda_comparison_df)